# 1:1 Replication — (Re-)Imag(in)ing Price Trends · Mac M4 Edition

**Fidelity guarantee:** The candlestick images and CNN are exactly as prescribed in the paper.
**Apple Silicon Optimization:** Designed specifically for M-series chips. Uses `mps` device to transparently max out all GPU cores via Apple Metal.
**Storage:** Uses the pure memmap architecture. Bypasses HDF5 to read raw `uint8` image bytes directly from SSD into Unified Memory via DMA.
**Persistent Cache:** Disk cleanup is disabled. The ~68GB `.dat` files persist locally. Re-running this notebook skips generation and instantly starts training.
**Portfolio Scoring Fix:** Strictly enforces non-overlapping evaluation for the 20 and 60 day horizons to perfectly match the paper's scorecard.


In [1]:
import os, tempfile, multiprocessing as mp
try: mp.set_start_method('fork', force=True)
except RuntimeError: pass
import concurrent.futures as cf
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

# Metal Performance Shaders (Apple Silicon) Setup
if torch.backends.mps.is_available():
    DEVICE_NAME = 'mps'
elif torch.cuda.is_available():
    DEVICE_NAME = 'cuda'
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE_NAME = 'cpu'

CPU = mp.cpu_count()

# ---- knobs ----
BATCH_SIZE   = 128
EPOCHS       = 100
LR           = 1e-5
PATIENCE     = 2
NUM_WORKERS  = 4  # Good default for local Mac SSDs
USE_AMP      = False 
MAX_PARALLEL = 1  # ALWAYS 1 for Mac MPS! Let Metal handle GPU parallelization under the hood.

# --- ensemble diversity ---
BAGGING           = True
TRAIN_FRAC        = 0.7
SHARED_SPLIT_SEED = 42

CACHE_DIR = './kaggle_dataset'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Device: {DEVICE_NAME} | CPU Cores: {CPU} | Cache: {CACHE_DIR} | Max Parallel: {MAX_PARALLEL}")


Device: mps | CPU Cores: 10 | Cache: ./kaggle_dataset | Max Parallel: 1


## 1. Data loading


In [2]:
TESTING = False

# Make sure to point this to where your parquet file is locally downloaded
file_name = './market_data_sp500_1997_2023.parquet'
if not os.path.exists(file_name):
    print(f"WARNING: {file_name} not found! Please download the dataset.")

df_market = pd.read_parquet(file_name) if os.path.exists(file_name) else pd.DataFrame()
if TESTING and not df_market.empty:
    keep = df_market['Ticker'].unique()[:3]
    df_market = df_market[df_market['Ticker'].isin(keep)].copy()


## 2. Image generation — exact original PIL (returned as uint8)


In [3]:
def _height(n_days): return 32 if n_days==5 else (64 if n_days==20 else 96)

def generate_image_uint8(window_df, n_days):
    height = _height(n_days); width = n_days*3
    img = Image.new('L', (width, height), color=0)
    draw = ImageDraw.Draw(img)
    if len(window_df)==0: return np.zeros((1,height,width), dtype=np.uint8)
    first_close = window_df['Close'].iloc[0]
    if first_close==0 or np.isnan(first_close): return np.zeros((1,height,width), dtype=np.uint8)
    prices  = window_df[['Open','High','Low','Close']].values / first_close
    ma      = window_df['Close'].rolling(window=n_days, min_periods=1).mean().values / first_close
    volumes = window_df['Volume'].values
    max_p, min_p = np.nanmax(prices), np.nanmin(prices); max_v = np.nanmax(volumes)
    if max_p==min_p: max_p = min_p + 1e-5
    if max_v==0:     max_v = 1e-5
    price_h = int(height*0.8); vol_h = height - price_h
    def p2y(p): return int(np.clip(price_h-1-(p-min_p)/(max_p-min_p)*(price_h-1),0,price_h-1))
    def v2y(v): return int(np.clip(height-1-(v/max_v)*(vol_h-1),price_h,height-1))
    ma_pts=[]
    for i in range(len(window_df)):
        x=i*3+1
        oy,hy,ly,cy = p2y(prices[i,0]),p2y(prices[i,1]),p2y(prices[i,2]),p2y(prices[i,3])
        draw.line([(x,hy),(x,ly)],fill=255,width=1)
        draw.line([(x-1,oy),(x,oy)],fill=255,width=1)
        draw.line([(x,cy),(x+1,cy)],fill=255,width=1)
        draw.line([(x,height-1),(x,v2y(volumes[i]))],fill=255,width=1)
        ma_pts.append((x,p2y(ma[i])))
    if len(ma_pts)>1: draw.line(ma_pts,fill=255,width=1)
    return np.array(img, dtype=np.uint8)


## 3. Pure Memmap Data Store (2-Pass Generation)
If the `.dat` file exists, it skips generation entirely.


In [4]:
REQUIRED = ['Open','High','Low','Close','Volume','MOM','STR','WSTR','TREND']

def _index_ticker(args):
    ticker, t_df, n_days = args
    t_df = t_df.sort_values('Date').reset_index(drop=True)
    nan_count = t_df[REQUIRED].isna().rolling(window=n_days).sum().sum(axis=1)
    valid = np.where(nan_count.values == 0)[0]
    
    samples = []
    for idx in valid:
        samples.append({
            'ticker': ticker, 'end_idx': int(idx), 'end_date': str(t_df['Date'].iloc[idx].date()),
            'MOM': float(t_df['MOM'].iloc[idx]), 'STR': float(t_df['STR'].iloc[idx]),
            'WSTR': float(t_df['WSTR'].iloc[idx]), 'TREND': float(t_df['TREND'].iloc[idx])
        })
    return ticker, t_df, samples

_G = {}
def _render_memmap_chunk(rng):
    lo, hi = rng
    mm = np.memmap(_G['dat_path'], dtype=np.uint8, mode='r+', shape=_G['shape'])
    nd = _G['n_days']
    for i in range(lo, hi):
        s = _G['samples'][i]; tdf = _G['grouped'][s['ticker']]
        idx = s['end_idx']
        mm[i, 0] = generate_image_uint8(tdf.iloc[idx-nd+1:idx+1], nd)
    mm.flush()
    return hi - lo

def build_memmap_store(df, n_days, dat_path, meta_path):
    H, W = _height(n_days), n_days*3
    
    # Skip if perfectly cached
    if os.path.exists(dat_path) and os.path.exists(meta_path):
        meta_df = pd.read_parquet(meta_path)
        N = len(meta_df)
        shape = (N, 1, H, W)
        print(f"  [CACHE HIT] Loaded {N:,} metadata records from {meta_path}. Image parsing skipped.")
        
        # We still need the original dataframe grouped by ticker to extract forward returns
        grouped = {t: df[df['Ticker']==t].copy().sort_values('Date').reset_index(drop=True) 
                   for t in meta_df['ticker'].unique()}
        return {'dat_path': dat_path, 'shape': shape, 'meta_df': meta_df, 'grouped': grouped}
    
    print(f"  [CACHE MISS] Generating new images for N={n_days}...")
    
    # PASS 1: Indexing
    args = [(t, df[df['Ticker']==t].copy(), n_days) for t in df['Ticker'].unique()]
    grouped, samples = {}, []
    
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        for tk, tdf, s in tqdm(ex.map(_index_ticker, args), total=len(args), desc=f"Index N={n_days}", leave=False):
            grouped[tk] = tdf
            samples.extend(s)
            
    N = len(samples)
    if N == 0: return None
    
    # Pre-allocate uncompressed dat file
    np.memmap(dat_path, dtype=np.uint8, mode='w+', shape=(N, 1, H, W)).flush()
    
    # Save metadata
    meta_df = pd.DataFrame(samples)
    meta_df.to_parquet(meta_path)
    
    # PASS 2: Parallel Render via Shared Memmap
    _G.update(dat_path=dat_path, shape=(N, 1, H, W), n_days=n_days, grouped=grouped, samples=samples)
    
    chunk = max(1, N // (CPU * 4))
    ranges = [(i, min(i + chunk, N)) for i in range(0, N, chunk)]
    
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        list(tqdm(ex.map(_render_memmap_chunk, ranges), total=len(ranges), desc=f"Render N={n_days} (Memmap)", leave=False))
        
    mb = (N * H * W) / 1e6
    print(f"  built memmap {dat_path}: {N:,} images (~{mb:,.0f} MB)")
    
    return {'dat_path': dat_path, 'shape': (N, 1, H, W), 'meta_df': meta_df, 'grouped': grouped}


## 4. Horizons, Splits, and DMA Dataset


In [5]:
def label_for_horizon(store, horizon):
    meta_df = store['meta_df']
    grouped = store['grouped']
    
    # Calculate forward returns dynamically
    fwd_rets = np.full(len(meta_df), np.nan)
    for t in meta_df['ticker'].unique():
        t_mask = meta_df['ticker'] == t
        t_meta = meta_df[t_mask]
        t_df = grouped[t]
        
        ei = t_meta['end_idx'].values
        valid_fwd = (ei + horizon) < len(t_df)
        
        base_prices = t_df['Close'].values[ei]
        fwd_prices = np.where(valid_fwd, t_df['Close'].values[np.minimum(ei+horizon, len(t_df)-1)], np.nan)
        
        fwd_rets[t_mask] = (fwd_prices / base_prices) - 1.0
        
    meta_df['Fwd_Ret'] = fwd_rets
    valid = ~np.isnan(meta_df['Fwd_Ret'])
    
    rows = np.where(valid)[0]
    label = (meta_df['Fwd_Ret'].values[rows] > 0).astype(np.int64)
    base = meta_df.iloc[rows].copy().rename(columns={'ticker': 'Ticker', 'end_date': 'Date'})
    
    return rows, label, base

def trainval_positions(base_df):
    d = base_df['Date'].values
    return np.where((d>='1997-01-01') & (d<='2004-12-31'))[0]

def test_positions(base_df):
    d = base_df['Date'].values
    te = np.where((d>='2005-01-01') & (d<='2023-12-31'))[0]
    return te[np.argsort(d[te], kind='stable')]

def subsample_split(tv_positions, seed, train_frac=0.7):
    tv = tv_positions.copy()
    np.random.default_rng(seed).shuffle(tv)
    cut = int(train_frac*len(tv))
    return tv[:cut], tv[cut:]

class MemmapDataset(Dataset):
    def __init__(self, dat_path, shape, image_rows, labels=None):
        self.dat_path = dat_path
        self.shape = shape
        self.rows = np.asarray(image_rows)
        self.labels = labels
        self._mm = None
        
    def __len__(self): return len(self.rows)
    
    def __getitem__(self, i):
        if self._mm is None: 
            self._mm = np.memmap(self.dat_path, dtype=np.uint8, mode='r', shape=self.shape)
        
        # OS DMA read directly to RAM
        img = torch.from_numpy(np.ascontiguousarray(self._mm[int(self.rows[i])])) 
        if self.labels is None: return img
        return img, int(self.labels[i])


## 5. CNN architecture — verbatim


In [6]:
def xavier_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super().__init__()
        if n_days == 5:    blocks, v_stride, v_dilation = 2, 1, 1
        elif n_days == 20: blocks, v_stride, v_dilation = 3, 3, 2
        else:              blocks, v_stride, v_dilation = 4, 3, 3
        layers, in_c, out_c = [], 1, 64
        for i in range(blocks):
            stride   = (v_stride, 1) if i == 0 else (1, 1)
            dilation = (v_dilation, 1) if i == 0 else (1, 1)
            layers += [nn.Conv2d(in_c, out_c, (5,3), stride=stride, padding=(2,1), dilation=dilation),
                       nn.BatchNorm2d(out_c), nn.LeakyReLU(0.01), nn.MaxPool2d((2,1))]
            in_c, out_c = out_c, out_c*2
        self.features = nn.Sequential(*layers)
        t = torch.zeros(1,1,_height(n_days), n_days*3)
        flat = self.features(t).view(1,-1).size(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(flat, 2))
        self.apply(xavier_init)
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))


## 6. Training engine


In [7]:
def _mk_loader(ds, shuffle):
    # pin_memory is False because Mac uses Unified Memory (RAM and VRAM are shared)
    kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=False)
    if NUM_WORKERS > 0: kw.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(ds, **kw)

def train_model(model, train_loader, val_loader, dev):
    import time
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    # AMP is generally disabled for Mac MPS by default unless forced, float32 is fast enough here
    best, pc = float('inf'), 0
    ckpt = tempfile.NamedTemporaryFile(suffix='.pt', delete=False).name
    
    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train(); train_loss = 0.0
        for imgs, labels in train_loader:
            imgs   = imgs.to(dev, non_blocking=True).float().div_(255.0)
            labels = labels.to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(imgs), labels)
            loss.backward(); optimizer.step()
            train_loss += loss.item()
        train_loss /= max(1, len(train_loader))
                
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(dev, non_blocking=True).float().div_(255.0)
                labels = labels.to(dev, non_blocking=True)
                val_loss += criterion(model(imgs), labels).item()
        val_loss /= max(1, len(val_loader))
        
        elapsed = time.time() - t0
        if val_loss < best:
            best, pc = val_loss, 0
            torch.save(model.state_dict(), ckpt)
            print(f"      Epoch {epoch+1:02d} | T-Loss: {train_loss:.4f} | V-Loss: {val_loss:.4f} | Time: {elapsed:.1f}s | [BEST]")
        else:
            pc += 1
            print(f"      Epoch {epoch+1:02d} | T-Loss: {train_loss:.4f} | V-Loss: {val_loss:.4f} | Time: {elapsed:.1f}s | Patience: {pc}/{PATIENCE}")
            if pc >= PATIENCE:
                print(f"      Early stopping triggered after {epoch+1} epochs.")
                break
            
    model.load_state_dict(torch.load(ckpt, map_location=dev, weights_only=True))
    os.remove(ckpt)
    return model

def predict_probs(model, loader, dev):
    model.eval(); out=[]
    with torch.no_grad():
        for imgs in loader:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
            p = torch.softmax(model(imgs), 1)[:, 1]
            out.append(p.float().cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def train_and_predict(job):
    n_days, store, tr_rows, tr_lab, va_rows, va_lab, te_rows, seed = job
    torch.manual_seed(seed)
    dev = torch.device(DEVICE_NAME)
    
    path, shape = store['dat_path'], store['shape']
    tl = _mk_loader(MemmapDataset(path, shape, tr_rows, tr_lab), True)
    vl = _mk_loader(MemmapDataset(path, shape, va_rows, va_lab), False)
    
    model = train_model(TrendCNN(n_days).to(dev), tl, vl, dev)
    te_l = _mk_loader(MemmapDataset(path, shape, te_rows), False)
    return predict_probs(model, te_l, dev)


## 7. Experiment loop


In [8]:
IMAGE_SIZES, HORIZONS, ENSEMBLE_SIZE = [5, 20, 60], [5, 20, 60], 5
if TESTING:
    IMAGE_SIZES, HORIZONS, ENSEMBLE_SIZE = [5], [5], 2

def calc_sharpe(signal_col, df, horizon):
    try:
        # STRICT PAPER EVALUATION: Enforce non-overlapping returns.
        dates = np.sort(df['Date'].unique())
        non_overlap_dates = dates[::horizon] 
        df_sub = df[df['Date'].isin(non_overlap_dates)].copy()
        
        df_sub['Decile'] = df_sub.groupby('Date')[signal_col].transform(
            lambda x: pd.qcut(x.rank(method='first'), 10, labels=False))
            
        high = df_sub[df_sub['Decile']==9].groupby('Date')['Fwd_Ret'].mean()
        low  = df_sub[df_sub['Decile']==0].groupby('Date')['Fwd_Ret'].mean()
        spread = high - low
        
        periods_per_year = 252 / horizon
        annual_spread = spread.mean() * periods_per_year
        annual_vol    = spread.std() * np.sqrt(periods_per_year)
        
        return annual_spread, (annual_spread/annual_vol if annual_vol != 0 else 0)
    except Exception:
        return 0, 0

results = []
for n_days in IMAGE_SIZES:
    dat_path = os.path.join(CACHE_DIR, f'raw_n{n_days}.dat')
    meta_path = os.path.join(CACHE_DIR, f'meta_n{n_days}.parquet')
    
    store = build_memmap_store(df_market, n_days, dat_path, meta_path)
    if store is None: continue

    for horizon in HORIZONS:
        print(f"\n{'='*54}\nConfig I{n_days} / R{horizon}\n{'='*54}")
        rows, label, base = label_for_horizon(store, horizon)
        te     = test_positions(base)
        tv_all = trainval_positions(base)
        if len(tv_all)==0 or len(te)==0: continue
            
        te_rows   = rows[te]
        eval_base = base.iloc[te].reset_index(drop=True)
  
        jobs = []
        for i in range(ENSEMBLE_SIZE):
            seed = 1000 + i
            split_seed = seed if BAGGING else SHARED_SPLIT_SEED
            tr, va = subsample_split(tv_all, split_seed, TRAIN_FRAC)
            jobs.append((n_days, store, rows[tr], label[tr], rows[va], label[va], te_rows, seed))

        preds = [None]*ENSEMBLE_SIZE
        # Mac strictly uses MAX_PARALLEL = 1 (Sequential loop)
        for k, j in enumerate(jobs):
            print(f"  --- model {k+1}/{ENSEMBLE_SIZE} on {DEVICE_NAME} ---")
            preds[k] = train_and_predict(j)

        eval_base['CNN_Prob'] = np.mean(preds, axis=0)
        for strat in ['CNN_Prob','MOM','STR','WSTR','TREND']:
            ret, sharpe = calc_sharpe(strat, eval_base.copy(), horizon)
            results.append({'Config':f"I{n_days}/R{horizon}", 'Strategy':strat, 'Ann_Return':ret, 'Ann_Sharpe':sharpe})
            
    # NOTE: os.remove(dat_path) has been intentionally removed! 
    # Your Mac SSD has plenty of space, so keeping the 68GB cache saves massive time on restarts.

df_results = pd.DataFrame(results)
display(df_results)
df_results.to_csv("final_experiment_results_mac.csv", index=False)
print("\nDone. Results -> final_experiment_results_mac.csv")


  [CACHE HIT] Loaded 2,739,387 metadata records from ./kaggle_dataset/meta_n5.parquet. Image parsing skipped.

Config I5 / R5
  --- model 1/5 on mps ---
      Epoch 01 | T-Loss: 0.9419 | V-Loss: 0.7042 | Time: 74.0s | [BEST]
      Epoch 02 | T-Loss: 0.8287 | V-Loss: 0.6967 | Time: 73.2s | [BEST]
      Epoch 03 | T-Loss: 0.7733 | V-Loss: 0.6938 | Time: 73.2s | [BEST]
      Epoch 04 | T-Loss: 0.7447 | V-Loss: 0.6935 | Time: 73.1s | [BEST]
      Epoch 05 | T-Loss: 0.7256 | V-Loss: 0.6926 | Time: 71.9s | [BEST]
      Epoch 06 | T-Loss: 0.7149 | V-Loss: 0.6920 | Time: 71.0s | [BEST]
      Epoch 07 | T-Loss: 0.7080 | V-Loss: 0.6945 | Time: 70.5s | Patience: 1/2
      Epoch 08 | T-Loss: 0.7026 | V-Loss: 0.6977 | Time: 70.1s | Patience: 2/2
      Early stopping triggered after 8 epochs.
  --- model 2/5 on mps ---
      Epoch 01 | T-Loss: 0.9446 | V-Loss: 0.7043 | Time: 70.4s | [BEST]
      Epoch 02 | T-Loss: 0.8277 | V-Loss: 0.6968 | Time: 70.1s | [BEST]
      Epoch 03 | T-Loss: 0.7718 | V-Los

KeyboardInterrupt: 

In [9]:
df_results = pd.DataFrame(results)
display(df_results)

,Config,Strategy,Ann_Return,Ann_Sharpe
0,I5/R5,CNN_Prob,0.122889,1.185263
1,I5/R5,MOM,-0.043598,-0.184956
2,I5/R5,STR,0.079244,0.368192
3,I5/R5,WSTR,0.110012,0.555862
4,I5/R5,TREND,-0.112267,-0.770808
5,I5/R20,CNN_Prob,0.023454,0.236541
6,I5/R20,MOM,-0.049448,-0.209852
7,I5/R20,STR,0.061945,0.305448
8,I5/R20,WSTR,0.042002,0.214950
9,I5/R20,TREND,-0.067741,-0.480842


In [10]:
df_results = pd.DataFrame(results)
display(df_results)
df_results.to_csv("final_experiment_results_mac.csv", index=False)
print("\nDone. Results -> final_experiment_results_mac.csv")

,Config,Strategy,Ann_Return,Ann_Sharpe
0,I5/R5,CNN_Prob,0.122889,1.185263
1,I5/R5,MOM,-0.043598,-0.184956
2,I5/R5,STR,0.079244,0.368192
3,I5/R5,WSTR,0.110012,0.555862
4,I5/R5,TREND,-0.112267,-0.770808
5,I5/R20,CNN_Prob,0.023454,0.236541
6,I5/R20,MOM,-0.049448,-0.209852
7,I5/R20,STR,0.061945,0.305448
8,I5/R20,WSTR,0.042002,0.214950
9,I5/R20,TREND,-0.067741,-0.480842



Done. Results -> final_experiment_results_mac.csv
